In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm 
import scipy.stats as stats
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates
from statsmodels.formula.api import ols

In [ ]:
fpath = 'C:\\Users\\gg00642\\OneDrive - University of Surrey\\Desktop\\Actigraphy Sara'
#fpath = 'C:\\Users\\gianl\\Desktop\\Actigraphy Sara'

__Mean exposure to light__

In [ ]:
df = pd.read_excel(fpath + '\\v3_lightdataset_avg.xlsx', sheet_name = 'Mean_light_daily')

In [ ]:
#in column timerange_0night, 0 is day, 1 is night 
#rename as day and night
#df['timerange_0night'] = df['timerange_0night'].replace({0: 'night', 1: 'day'})
df['location_0_ita'] = df['location_0_ita'].replace({0: 'ita', 1: 'uk'})

In [ ]:
df_daylight1 = pd.read_excel(fpath + '\\1.0_sunset_sunrise_UTC.xlsx', sheet_name='Sheet1')

In [ ]:
def adjust_value(row):
    timeshift = row['DST(0=ST)']
    
    if timeshift == 1:
                return row['sunrise (uk), hours'] + 1, row['sunset (uk), hours'] + 1, row['sunrise (ita), hours'] + 2, row['sunset (ita), hours'] + 2
    elif timeshift == 0:
                return row['sunrise (uk), hours'] - 0, row['sunset (uk), hours'] - 0, row['sunrise (ita), hours'] + 1, row['sunset (ita), hours'] + 1
    
    return row['sunrise (uk), hours'], row['sunset (uk), hours'], row['sunrise (ita), hours'], row['sunset (ita), hours']

df_daylight1[['sunrise (uk), hours_adjust', 'sunset (uk), hours_adjust', 'sunrise (ita), hours_adjust', 'sunset (ita), hours_adjust']] = df_daylight1.apply(adjust_value, axis=1, result_type='expand')

In [ ]:
# Add a third y-axis for daylight length
fig, ax1 = plt.subplots(figsize=(25, 8))

# Remove spines
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

# Add gray background
ax1.set_facecolor('#f0f0f0')

# sleep onset and offset
sns.scatterplot(x='date', y='mean_light', hue='location_0_ita', style='weekday_type', data=df, palette=['darkblue', 'darkorange'], legend='full', ax=ax1)


# Adding labels and title
ax1.set_xlabel('')
ax1.set_ylabel('Mean light (log lux)', fontsize=14)
ax1.set_title('Mean light in UK and Italy over time\n', fontsize=16)

# Remove space y axis and plot
plt.gca().margins(x=0)

# Remove space y axis and plot
ax1.margins(x=0)

ax1.legend(loc='upper left', bbox_to_anchor=(1.01, 0.5)) 

# add x axis ticks every 30 days and turn of 45 degrees
ax1.xaxis.set_major_locator(mdates.DayLocator(interval=30))
plt.xticks(rotation=45)

# Add vertical lines and text for DST and ST
for date, label in [('2022-10-30', 'start ST'), ('2023-10-29', 'start ST'), ('2023-03-26', 'start DST'), ('2024-03-31', 'start DST'), ('2024-10-27', 'start ST')]:
    plt.axvline(x=pd.to_datetime(date), color='dimgrey', linestyle='--')
    plt.text(pd.to_datetime(date), plt.ylim()[0], label, ha='right', va='bottom', color='dimgray')

plt.xlim(pd.to_datetime('2022-09-19'), pd.to_datetime('2025-01-01'))

plt.show()

In [ ]:
# Plot the mean light data over time, marking night and day, grouped by season and weekday_type
#plt.figure(figsize=(15, 10))
#sns.scatterplot(x='date_start', y='mean_light', hue='timerange_0night', size='weekday_type', data=df, sizes=(10, 50))
#plt.xlabel('Date')
#plt.ylabel('Mean Light')
#plt.title('Mean Light over Time (Day vs Night) with Weekday Type')
#plt.legend(title='Time Range')
#plt.legend(title='Time Range', bbox_to_anchor=(1.05, 1.02), loc='upper left')

#plt.show()

In [ ]:
#distribution of mean 
plt.figure(figsize=(10,6))
sns.histplot(data=df, x='mean_light', kde = False, bins = 50, hue='location_0_ita')
plt.xlabel('Mean light (log lux)')
plt.ylabel('Frequency')
plt.title('Distribution of mean light')
plt.show()

In [ ]:
# remove light les than 0.1
df = df[df['mean_light'] > 0.1]

In [ ]:
shapiro_test = stats.shapiro(df['mean_light'])
#shapiro_test_day = stats.shapiro(df_day['mean_light'])   
#shapiro_test_night = stats.shapiro(df_night['mean_light'])

In [ ]:
shapiro_test #, shapiro_test_day, shapiro_test_night

In [ ]:
t_test_result = stats.ttest_ind(df[df['location_0_ita'] == 'ita']['mean_light'], df[df['location_0_ita'] == 'uk']['mean_light'])

t_test_result

In [ ]:
# Descriptive statistics for 'mean_light' by location
df_mean_light = df.groupby('location_0_ita')['mean_light'].describe()

df_mean_light

In [ ]:
# Regression analysis for day
model = smf.ols(
    'mean_light ~ C(location_0_ita) + photoperiod + C(weekday_type)',
    data=df
).fit()

# Display the regression summaries
model.summary()

In [ ]:
# perform a multifactors ANOVA to test the effect of location, season and weekday_type on TATp10_min
model_df = ols('mean_light ~ C(location_0_ita) + photoperiod + C(weekday_type)', data=df).fit()
anova_df = sm.stats.anova_lm(model_df, typ=3)
anova_df

In [ ]:
# mean and std of mean light by location and weekday_type

In [ ]:
# coefficient of the anoav model
model_df.params 


Seasons

In [ ]:
df_summer = df[df['season'] == 'Summer']

df_winter = df[df['season'] == 'Winter']

-In Summer

In [ ]:
utest_loc_s = stats.mannwhitneyu(df_summer[df_summer['location_0_ita'] == 'ita']['mean_light'], df_summer[df_summer['location_0_ita'] == 'uk']['mean_light'])

utest_loc_s

In [ ]:
summer_loc = df_summer.groupby('location_0_ita')['mean_light'].describe()
summer_loc

-In Winter

In [ ]:
utest_loc_w = stats.mannwhitneyu(df_winter[df_winter['location_0_ita'] == 'ita']['mean_light'], df_winter[df_winter['location_0_ita'] == 'uk']['mean_light'])

utest_loc_w

In [ ]:
winter_loc = df_winter.groupby('location_0_ita')['mean_light'].describe()
winter_loc

_Mean light exposure 7:00-17:00_

In [ ]:
df_range = pd.read_excel(fpath + '\\v3_lightdataset_avg.xlsx', sheet_name = '7_17_meanlight')

In [ ]:
#in column timerange_0night, 0 is day, 1 is night 
#rename as day and night
#df['timerange_0night'] = df['timerange_0night'].replace({0: 'night', 1: 'day'})
df_range['location_0_ita'] = df_range['location_0_ita'].replace({0: 'ita', 1: 'uk'})

In [ ]:
df_daylight1 = pd.read_excel(fpath + '\\1.0_sunset_sunrise_UTC.xlsx', sheet_name='Sheet1')

In [ ]:
def adjust_value(row):
    timeshift = row['DST(0=ST)']
    
    if timeshift == 1:
                return row['sunrise (uk), hours'] + 1, row['sunset (uk), hours'] + 1, row['sunrise (ita), hours'] + 2, row['sunset (ita), hours'] + 2
    elif timeshift == 0:
                return row['sunrise (uk), hours'] - 0, row['sunset (uk), hours'] - 0, row['sunrise (ita), hours'] + 1, row['sunset (ita), hours'] + 1
    
    return row['sunrise (uk), hours'], row['sunset (uk), hours'], row['sunrise (ita), hours'], row['sunset (ita), hours']

df_daylight1[['sunrise (uk), hours_adjust', 'sunset (uk), hours_adjust', 'sunrise (ita), hours_adjust', 'sunset (ita), hours_adjust']] = df_daylight1.apply(adjust_value, axis=1, result_type='expand')

In [ ]:
# Add a third y-axis for daylight length
fig, ax1 = plt.subplots(figsize=(25, 8))

# Remove spines
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

# Add gray background
ax1.set_facecolor('#f0f0f0')

# sleep onset and offset
sns.scatterplot(x='date', y='mean_light', hue='location_0_ita', style='weekday_type', data=df_range, palette=['darkblue', 'darkorange'], legend='full', ax=ax1)


# Adding labels and title
ax1.set_xlabel('')
ax1.set_ylabel('Mean light (log lux)', fontsize=14)
ax1.set_title('Mean light in UK and Italy over time (range 07:00-17:00)\n', fontsize=16)

# Remove space y axis and plot
plt.gca().margins(x=0)

# Remove space y axis and plot
ax1.margins(x=0)

# add x axis ticks every 30 days and turn of 45 degrees
ax1.xaxis.set_major_locator(mdates.DayLocator(interval=30))
plt.xticks(rotation=45)

# Add vertical lines and text for DST and ST
for date, label in [('2022-10-30', 'start ST'), ('2023-10-29', 'start ST'), ('2023-03-26', 'start DST'), ('2024-03-31', 'start DST'), ('2024-10-27', 'start ST')]:
    plt.axvline(x=pd.to_datetime(date), color='dimgrey', linestyle='--')
    plt.text(pd.to_datetime(date), plt.ylim()[0], label, ha='right', va='bottom', color='dimgray')

plt.xlim(pd.to_datetime('2022-09-19'), pd.to_datetime('2025-01-01'))

plt.show()

In [ ]:
# Plot the mean light data over time, marking night and day, grouped by season and weekday_type
#plt.figure(figsize=(15, 10))
#sns.scatterplot(x='date_start', y='mean_light', hue='timerange_0night', size='weekday_type', data=df, sizes=(10, 50))
#plt.xlabel('Date')
#plt.ylabel('Mean Light')
#plt.title('Mean Light over Time (Day vs Night) with Weekday Type')
#plt.legend(title='Time Range')
#plt.legend(title='Time Range', bbox_to_anchor=(1.05, 1.02), loc='upper left')

#plt.show()

In [ ]:
#distribution of mean 
plt.figure(figsize=(10,6))
sns.histplot(data=df_range, x='mean_light', kde = False, bins = 50, hue='location_0_ita')
plt.xlabel('Mean light (log lux)')
plt.ylabel('Frequency')
plt.title('Distribution of mean light')
plt.show()

In [ ]:
# remove light les than 0.1
df_range = df_range[df_range['mean_light'] > 0.1]

In [ ]:
shapiro_test = stats.shapiro(df_range['mean_light'].dropna())
#shapiro_test_day = stats.shapiro(df_day['mean_light'])   
#shapiro_test_night = stats.shapiro(df_night['mean_light'])

In [ ]:
shapiro_test #, shapiro_test_day, shapiro_test_night

In [ ]:
utest_location = stats.mannwhitneyu(df_range[df_range['location_0_ita'] == 'ita']['mean_light'].dropna(), df_range[df_range['location_0_ita'] == 'uk']['mean_light'].dropna())

utest_location

In [ ]:
# Descriptive statistics for 'mean_light' by location
df_mean_light = df_range.groupby('location_0_ita')['mean_light'].describe()

df_mean_light

In [ ]:
# Regression analysis for day
model = smf.ols(
    'mean_light ~ C(location_0_ita) + photoperiod + C(weekday_type)',
    data=df_range
).fit()

# Display the regression summaries
model.summary()

In [ ]:
# perform a multifactors ANOVA to test the effect of location, season and weekday_type on TATp10_min
model_df_range = ols('mean_light ~ C(location_0_ita) + photoperiod + C(weekday_type)', data=df_range).fit()
anova_df_range = sm.stats.anova_lm(model_df_range, typ=3)
anova_df_range

In [ ]:
# coefficient of the anoav model
model_df_range.params 


__Time above threshold (10-100-500-1000-2500)__

_TATp10_

In [ ]:
df_TATp10 = pd.read_excel(fpath + '\\v3_lightdataset_avg.xlsx', sheet_name = 'TATp_10')

In [ ]:
# remove outliers according to 3 standard deviations
df_TATp10 = df_TATp10[np.abs(df_TATp10['TATp10_min'] - df_TATp10['TATp10_min'].mean()) <= (3 * df_TATp10['TATp10_min'].std())]

In [ ]:
df_TATp10['location_0_ita'] = df_TATp10['location_0_ita'].replace({0: 'ita', 1: 'uk'})

In [ ]:
#distribution of mean 
plt.figure(figsize=(10,6))
sns.histplot(data=df_TATp10, x='TATp10_min', kde = False, bins = 50, hue='location_0_ita')
plt.xlabel('TATp10_min')
plt.ylabel('Frequency')
plt.title('Distribution of TATp10_min')
plt.show()

In [ ]:
shapiro_test_tatp10 = stats.shapiro(df_TATp10['TATp10_min'])

shapiro_test_tatp10

In [ ]:
utest_location_tatp10 = stats.mannwhitneyu(df_TATp10[df_TATp10['location_0_ita'] == 'ita']['TATp10_min'], df_TATp10[df_TATp10['location_0_ita'] == 'uk']['TATp10_min'])

utest_location_tatp10

In [ ]:
# grpup by location and mean,std,min and max of 'TATp10_min
df_TATp10_loc = df_TATp10.groupby('location_0_ita').agg({'TATp10_min': ['mean', 'std', 'min', 'max']})
df_TATp10_loc

In [ ]:
# perform a multifactors ANOVA to test the effect of location, season and weekday_type on TATp10_min
model_tatp10 = ols('TATp10_min ~ C(location_0_ita) + photoperiod + C(weekday_type)', data=df_TATp10).fit()
anova_table = sm.stats.anova_lm(model_tatp10, typ=3)
anova_table

In [ ]:
# coefficient of the anoav model
model_tatp10.params 


In [ ]:
# Regression analysis for night
model_TATp10 = smf.ols(
    "TATp10_min ~ C(location_0_ita) + photoperiod + C(weekday_type)",
    data=df_TATp10
).fit()

model_TATp10.summary()

In [ ]:
#The coefficient estimate in the output indicate the average change in the log odds of the response variable associated with a one unit increase in each predictor variable.

In [ ]:
# Add a third y-axis for daylight length
fig, ax1 = plt.subplots(figsize=(25, 8))

# Remove spines
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

# Add gray background
ax1.set_facecolor('#f0f0f0')

# sleep onset and offset
sns.scatterplot(x='date', y='TATp10_min', hue='location_0_ita', style='weekday_type', data=df_TATp10, palette=['darkblue', 'darkorange'], legend='full', ax=ax1)

# Adding labels and title
ax1.set_xlabel('')
ax1.set_ylabel('TAT (min)', fontsize=14)
ax1.set_title('Time above 10 lux in UK and Italy over time\n', fontsize=16)

# Remove space y axis and plot
plt.gca().margins(x=0)

# Remove space y axis and plot
ax1.margins(x=0)

# add x axis ticks every 30 days and turn of 45 degrees
ax1.xaxis.set_major_locator(mdates.DayLocator(interval=30))
plt.xticks(rotation=45)

# Add vertical lines and text for DST and ST
for date, label in [('2022-10-30', 'start ST'), ('2023-10-29', 'start ST'), ('2023-03-26', 'start DST'), ('2024-03-31', 'start DST'), ('2024-10-27', 'start ST')]:
    plt.axvline(x=pd.to_datetime(date), color='dimgrey', linestyle='--')
    plt.text(pd.to_datetime(date), plt.ylim()[0], label, ha='right', va='bottom', color='dimgray')

plt.xlim(pd.to_datetime('2022-09-19'), pd.to_datetime('2025-01-01'))

plt.show()

_TATp100_

In [ ]:
df_TATp100 = pd.read_excel(fpath + '\\v3_lightdataset_avg.xlsx', sheet_name = 'TATp_100')

In [ ]:
# remove outliers according to 3 standard deviations
#df_TATp100 = df_TATp100[np.abs(df_TATp100['TATp100_min'] - df_TATp100['TATp100_min'].mean()) <= (3 * df_TATp100['TATp100_min'].std())]

In [ ]:
#distribution of mean 
plt.figure(figsize=(10,6))
sns.histplot(data=df_TATp100, x='TATp100_min', kde = False, bins = 50, hue='location_0_ita')
plt.xlabel('TATp100_min')
plt.ylabel('Frequency')
plt.title('Distribution of TATp100_min')
plt.show()

In [ ]:
shapiro_test_tatp100 = stats.shapiro(df_TATp100['TATp100_min'])

shapiro_test_tatp100

In [ ]:
df_TATp100['location_0_ita'] = df_TATp100['location_0_ita'].replace({0: 'ita', 1: 'uk'})

In [ ]:
utest_location_tatp100 = stats.mannwhitneyu(df_TATp100[df_TATp100['location_0_ita'] == 'ita']['TATp100_min'], df_TATp100[df_TATp100['location_0_ita'] == 'uk']['TATp100_min'])

utest_location_tatp100

In [ ]:
# by location and mean,std,min and max of 'TATp100_min
df_TATp100_loc = df_TATp100.groupby('location_0_ita').agg({'TATp100_min': ['mean', 'std', 'min', 'max']})
df_TATp100_loc

In [ ]:
# Regression analysis for night
model_TATp100 = smf.ols(
    "TATp100_min ~ C(location_0_ita) + photoperiod + C(weekday_type)",
    data=df_TATp100
).fit()

model_TATp100.summary()

In [ ]:
# Add a third y-axis for daylight length
fig, ax1 = plt.subplots(figsize=(25, 8))

# Remove spines
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

# Add gray background
ax1.set_facecolor('#f0f0f0')

# sleep onset and offset
sns.scatterplot(x='date', y='TATp100_min', hue='location_0_ita', style='weekday_type', data=df_TATp100, palette=['darkblue', 'darkorange'], legend='full', ax=ax1)

# Adding labels and title
ax1.set_xlabel('')
ax1.set_ylabel('TAT (min)', fontsize=14)
ax1.set_title('Time above 100 lux in UK and Italy over time\n', fontsize=16)

# Remove space y axis and plot
plt.gca().margins(x=0)

# Remove space y axis and plot
ax1.margins(x=0)

# add x axis ticks every 30 days and turn of 45 degrees
ax1.xaxis.set_major_locator(mdates.DayLocator(interval=30))
plt.xticks(rotation=45)

# Add vertical lines and text for DST and ST
for date, label in [('2022-10-30', 'start ST'), ('2023-10-29', 'start ST'), ('2023-03-26', 'start DST'), ('2024-03-31', 'start DST'), ('2024-10-27', 'start ST')]:
    plt.axvline(x=pd.to_datetime(date), color='dimgrey', linestyle='--')
    plt.text(pd.to_datetime(date), plt.ylim()[0], label, ha='right', va='bottom', color='dimgray')

plt.xlim(pd.to_datetime('2022-09-19'), pd.to_datetime('2025-01-01'))

plt.show()

_TATp500_

In [ ]:
df_TATp500 = pd.read_excel(fpath + '\\v3_lightdataset_avg.xlsx', sheet_name = 'TATp_500')

In [ ]:
# remove value less than 0
df_TATp500 = df_TATp500[df_TATp500['TATp500_min'] > 1]

In [ ]:
# remove outliers according to 3 standard deviations
#df_TATp490 = df_TATp490[np.abs(df_TATp490['TATp490_min'] - df_TATp490['TATp490_min'].mean()) <= (3 * df_TATp490['TATp490_min'].std())]

In [ ]:
df_TATp500['location_0_ita'] = df_TATp500['location_0_ita'].replace({0: 'ita', 1: 'uk'})

In [ ]:
#distribution of mean 
plt.figure(figsize=(10,6))
sns.histplot(data=df_TATp500, x='TATp500_min', kde = False, bins = 50, hue='location_0_ita')
plt.xlabel('TATp500_min')
plt.ylabel('Frequency')
plt.title('Distribution of TATp500_min')
plt.show()

In [ ]:
utest_location_tatp500 = stats.mannwhitneyu(df_TATp500[df_TATp500['location_0_ita'] == 'ita']['TATp500_min'], df_TATp500[df_TATp500['location_0_ita'] == 'uk']['TATp500_min'])

utest_location_tatp500

In [ ]:
# by location and mean,std,min and max of 'TATp500_min
df_TATp500_loc = df_TATp500.groupby('location_0_ita').agg({'TATp500_min': ['mean', 'median', 'std', 'min', 'max']})
df_TATp500_loc

In [ ]:
# Fix: Replace zero values in TATp500_min with a small positive value
df_TATp500["TATp500_min"] = df_TATp500["TATp500_min"].replace(0, 1e-6)

# Fit a GLM with Gamma distribution and correct log link function
glm_model_fixed = smf.glm(
    "TATp500_min ~ C(location_0_ita) + photoperiod + C(weekday_type)",
    data=df_TATp500,
    family=sm.families.Gamma(link=sm.families.links.Log())  # Fix deprecated log alias
).fit()

# Display the corrected regression summary
glm_model_fixed.summary()


In [ ]:
#convert the coefficients to the original scale and print them
coef = np.exp(glm_model_fixed.params)
print(coef)

In [ ]:
# convert the coef in percentage
coef = np.exp(glm_model_fixed.params) - 1
coef = coef * 100
print(coef)

In [ ]:
# Add a third y-axis for daylight length
fig, ax1 = plt.subplots(figsize=(25, 8))

# Remove spines
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

# Add gray background
ax1.set_facecolor('#f0f0f0')

# sleep onset and offset
sns.scatterplot(x='date', y='TATp500_min', hue='location_0_ita', style='weekday_type', data=df_TATp500, palette=['darkblue', 'darkorange'], legend='full', ax=ax1)

# Adding labels and title
ax1.set_xlabel('')
ax1.set_ylabel('TAT (min)', fontsize=14)
ax1.set_title('Time above 500 lux in UK and Italy over time\n', fontsize=16)

# Remove space y axis and plot
plt.gca().margins(x=0)

# Remove space y axis and plot
ax1.margins(x=0)

# add x axis ticks every 30 days and turn of 45 degrees
ax1.xaxis.set_major_locator(mdates.DayLocator(interval=30))
plt.xticks(rotation=45)

# Add vertical lines and text for DST and ST
for date, label in [('2022-10-30', 'start ST'), ('2023-10-29', 'start ST'), ('2023-03-26', 'start DST'), ('2024-03-31', 'start DST'), ('2024-10-27', 'start ST')]:
    plt.axvline(x=pd.to_datetime(date), color='dimgrey', linestyle='--')
    plt.text(pd.to_datetime(date), plt.ylim()[0], label, ha='right', va='bottom', color='dimgray')

plt.xlim(pd.to_datetime('2022-09-19'), pd.to_datetime('2025-01-01'))

plt.show()

_TATp1000_

In [ ]:
df_TATp1000 = pd.read_excel(fpath + '\\v3_lightdataset_avg.xlsx', sheet_name = 'TATp_1000')

In [ ]:
df_TATp1000['location_0_ita'] = df_TATp1000['location_0_ita'].replace({0: 'ita', 1: 'uk'})

In [ ]:
#distribution of mean 
plt.figure(figsize=(10,6))
sns.histplot(data=df_TATp1000, x='TATp1000_min', kde = False, bins = 50, hue='location_0_ita')
plt.xlabel('TATp1000_min')
plt.ylabel('Frequency')
plt.title('Distribution of TATp1000_min')
plt.show()

In [ ]:
shapiro_test_tatp1000 = stats.shapiro(df_TATp1000['TATp1000_min'])

shapiro_test_tatp1000

In [ ]:
utest_location_tatp1000 = stats.mannwhitneyu(df_TATp1000[df_TATp1000['location_0_ita'] == 'ita']['TATp1000_min'], df_TATp1000[df_TATp1000['location_0_ita'] == 'uk']['TATp1000_min'])

utest_location_tatp1000

In [ ]:
# by location and mean,std,min and max of 'TATp1000_min
df_TATp1000_loc = df_TATp1000.groupby('location_0_ita').agg({'TATp1000_min': ['mean', 'std', 'min', 'max']})
df_TATp1000_loc

In [ ]:
# Fix: Replace zero values in TATp1000_min with a small positive value
df_TATp1000["TATp1000_min"] = df_TATp1000["TATp1000_min"].replace(0, 1e-6)

# Fit a GLM with Gamma distribution and correct log link function
glm_model_fixed = smf.glm(
    "TATp1000_min ~ location_0_ita + photoperiod + C(weekday_type)",
    data=df_TATp1000,
    family=sm.families.Gamma(link=sm.families.links.Log())  # Fix deprecated log alias
).fit()

# Display the corrected regression summary
glm_model_fixed.summary()


In [ ]:
#convert the coefficients to the original scale and print them
coef = np.exp(glm_model_fixed.params)
print(coef)

In [ ]:
# convert the coef in percentage
coef = np.exp(glm_model_fixed.params) - 1
coef = coef * 100
print(coef)

In [ ]:
# Add a third y-axis for daylight length
fig, ax1 = plt.subplots(figsize=(25, 8))

# Remove spines
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

# Add gray background
ax1.set_facecolor('#f0f0f0')

# sleep onset and offset
sns.scatterplot(x='date', y='TATp1000_min', hue='location_0_ita', style='weekday_type', data=df_TATp1000, palette=['darkblue', 'darkorange'], legend='full', ax=ax1)

# Adding labels and title
ax1.set_xlabel('')
ax1.set_ylabel('TAT (min)', fontsize=14)
ax1.set_title('Time above 1000 lux in UK and Italy over time\n', fontsize=16)

# Remove space y axis and plot
plt.gca().margins(x=0)

# Remove space y axis and plot
ax1.margins(x=0)

# add x axis ticks every 30 days and turn of 45 degrees
ax1.xaxis.set_major_locator(mdates.DayLocator(interval=30))
plt.xticks(rotation=45)

# Add vertical lines and text for DST and ST
for date, label in [('2022-10-30', 'start ST'), ('2023-10-29', 'start ST'), ('2023-03-26', 'start DST'), ('2024-03-31', 'start DST'), ('2024-10-27', 'start ST')]:
    plt.axvline(x=pd.to_datetime(date), color='dimgrey', linestyle='--')
    plt.text(pd.to_datetime(date), plt.ylim()[0], label, ha='right', va='bottom', color='dimgray')

plt.xlim(pd.to_datetime('2022-09-19'), pd.to_datetime('2025-01-01'))

plt.show()

_TATp2500_

In [ ]:
df_TATp2500 = pd.read_excel(fpath + '\\v3_lightdataset_avg.xlsx', sheet_name = 'TATp_2500')

In [ ]:
df_TATp2500['location_0_ita'] = df_TATp2500['location_0_ita'].replace({0: 'ita', 1: 'uk'})

In [ ]:
#distribution of mean 
plt.figure(figsize=(10,6))
sns.histplot(data=df_TATp2500, x='TATp2500_min', kde = False, bins = 50, hue='location_0_ita')
plt.xlabel('TATp2500_min')
plt.ylabel('Frequency')
plt.title('Distribution of TATp2500_min')
plt.show()

In [ ]:
shapiro_test_tatp2500 = stats.shapiro(df_TATp2500['TATp2500_min'])

shapiro_test_tatp2500

In [ ]:
utest_location_tatp2500 = stats.mannwhitneyu(df_TATp2500[df_TATp2500['location_0_ita'] == 'ita']['TATp2500_min'], df_TATp2500[df_TATp2500['location_0_ita'] == 'uk']['TATp2500_min'])

utest_location_tatp2500

In [ ]:
# by location and mean,std,min and max of 'TATp2500_min
df_TATp2500_loc = df_TATp2500.groupby('location_0_ita').agg({'TATp2500_min': ['mean', 'std', 'min', 'max']})
df_TATp2500_loc

In [ ]:
# Fix: Replace zero values in TATp2500_min with a small positive value
df_TATp2500["TATp2500_min"] = df_TATp2500["TATp2500_min"].replace(0, 1e-6)

# Fit a GLM with Gamma distribution and correct log link function
glm_model_fixed = smf.glm(
    "TATp2500_min ~ C(location_0_ita) + photoperiod + C(weekday_type)",
    data=df_TATp2500,
    family=sm.families.Gamma(link=sm.families.links.Log())  # Fix deprecated log alias
).fit()

# Display the corrected regression summary
glm_model_fixed.summary()


In [ ]:
#convert the coefficients to the original scale and print them
coef = np.exp(glm_model_fixed.params)
print(coef)

In [ ]:
# convert the coef in percentage
coef = np.exp(glm_model_fixed.params) - 1
coef = coef * 100
print(coef)

In [ ]:
# Add a third y-axis for daylight length
fig, ax1 = plt.subplots(figsize=(25, 8))

# Remove spines
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

# Add gray background
ax1.set_facecolor('#f0f0f0')

# sleep onset and offset
sns.scatterplot(x='date', y='TATp2500_min', hue='location_0_ita', style='weekday_type', data=df_TATp2500, palette=['darkblue', 'darkorange'], legend='full', ax=ax1)

# Adding labels and title
ax1.set_xlabel('')
ax1.set_ylabel('TAT (min)', fontsize=14)
ax1.set_title('Time above 2500 lux in UK and Italy over time\n', fontsize=16)

# Remove space y axis and plot
plt.gca().margins(x=0)

# Remove space y axis and plot
ax1.margins(x=0)

# add x axis ticks every 30 days and turn of 45 degrees
ax1.xaxis.set_major_locator(mdates.DayLocator(interval=30))
plt.xticks(rotation=45)

# Add vertical lines and text for DST and ST
for date, label in [('2022-10-30', 'start ST'), ('2023-10-29', 'start ST'), ('2023-03-26', 'start DST'), ('2024-03-31', 'start DST'), ('2024-10-27', 'start ST')]:
    plt.axvline(x=pd.to_datetime(date), color='dimgrey', linestyle='--')
    plt.text(pd.to_datetime(date), plt.ylim()[0], label, ha='right', va='bottom', color='dimgray')

plt.xlim(pd.to_datetime('2022-09-19'), pd.to_datetime('2025-01-01'))

plt.show()